# Raw spectra review

Per-**device** look at the *raw* EIS spectra and FSCV voltammograms across all measured
timepoints — **run this first** to choose the quality-filtering stringency (especially
`BAND`) that you then set in `quality_filtering_dashboard.ipynb`.

Each channel is a panel; timepoints are overlaid **older faded → latest solid**. Unless a
section says otherwise, every trace is the **replicate-average** of a measurement (EIS: mean
over replicate spectra; FSCV: mean of the first `REPLICATES` cycles).

---

**Where this fits — the ElectroPyCal analysis pipeline** (`pip install electropycal`):

1. ➡️ **`raw_spectra_review` — inspect raw EIS/FSCV spectra; choose the analysis band & QC stringency  ← you are here**
2. `quality_filtering_dashboard` — set QC gates, see which `(channel, timepoint)` sensors pass, save the shared config
3. `stabilization_review` — confirm each channel's FSCV response stabilized (optional / parallel)
4. `diagnostics_review` — characterize the featureset before modeling — *is recalibration well-posed?*
5. `discovery_checkpointed` — run the model-discovery task queue batch-by-batch under nested CV
6. `discovery_results_review` — inspect a finished discovery run; pick a model
7. `deployment_domain_shift` — freeze the chosen model; recalibrate in-vivo data; monitor CORAL domain shift

Consistency knobs (band, QC gates) are shared across notebooks via `<ROOT>/electropycal_analysis_config.json`. See `docs/USAGE.md`.


## Setup — run first

Installs `electropycal` from PyPI. On Colab it also mounts Google Drive, so you can point
`ROOT` at raw data in your own Drive; skip the mount if your data is local.

```
pip install electropycal
```

After installing on Colab, do **Runtime → Restart session** so the new module loads.


In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q electropycal
    from google.colab import drive
    drive.mount("/content/drive")     # only needed if ROOT points into your Drive


In [ ]:
%matplotlib inline
import tempfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from electropycal.viz import set_pub_style
from electropycal.rawspectra import (RawSpectraIndex, plot_eis, plot_eis_replicate_spread,
    plot_inductive_onset_vs_time, plot_fscv_loops, plot_dose_response, plot_snr_vs_conc,
    plot_vpeak_vs_conc, plot_vs_time, onset_table, eis_spread_table, dose_stats_table,
    edge_pinning_table)
set_pub_style()

# --- point ROOT at your data folder (Google Drive) ---
ROOT = None   # <- your own PSTrace export directory; None uses the shipped demo
# demo_input() locates demo/ from ANY working directory -- the repo root, notebooks/,
# or anywhere else -- and synthesizes an equivalent tree when no checkout is reachable
# (e.g. Colab). A relative literal here breaks in whichever cwd it was not written for.
# ROOT must be a LOCAL FILESYSTEM PATH. A Drive or cloud share URL will not work:
# nothing here downloads for you. On Colab, mount the drive first and use the
# mounted path:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     ROOT = '/content/drive/MyDrive/<your-folder>'
# Point ROOT at the directory that DIRECTLY CONTAINS the <YYYYMMDD>_..._signal
# folders, not at its parent. If you are one level off, extraction now says so and
# names the directory it thinks you meant.
from electropycal._demo import demo_input
ROOT = demo_input("in_vitro", ROOT)
print("ROOT =", ROOT)

## Available data — signal measurements by device / date / timepoint
`timepoint` = elapsed days since each device's first **signal** session (`D0`, `D1`, …). The
**channels** column is read from the header of each session's `<deviceid>_fscv_0nM.csv` file.

> This scan is **fast** — the signal index is built from filenames alone (no file reads), and
> the channel list comes from the *header line* of the 0 nM files only (not the waveforms or the
> EIS/dose files). Heavy parsing happens lazily, per plot. Only **signal** folders are read.

In [ ]:
# One index over ROOT: filename scan + cached, replicate-averaged trace loaders + plots.
# All processing knobs come from the shared AnalysisConfig (band, peak method, smoothing, replicates).
IDX = RawSpectraIndex(ROOT)
IDX.print_settings()                    # ROOT + resolved defaults + how to customize
print()
avail = IDX.available_table()
print(f"{avail.device.nunique()} device(s), {len(avail)} signal sessions (0nM headers only):")
with pd.option_context("display.max_rows", None):   # show ALL rows
    display(IDX.label_devices(avail))

## Selection & parameters
Pick what to review by **device type**, **device id**, and **timepoint** — each defaults to *all
available*. `DEVICE_TYPES` selects `neurostring` / `cfme` / …; `DEVICES` selects device ids (within
those types); `TIMEPOINTS` selects day integers (`D0`=0, `D1`=1, …). The rest are display / analysis
knobs, surfaced so you can tune what is drawn.

In [ ]:
# SELECTION is local to this run; PARAMETERS come from the shared config. Override a parameter by
# rebuilding, e.g. RawSpectraIndex(ROOT, peak_method='chord', smooth_window=0) — see settings print above.
DEVICE_TYPES = ["neurostring"]      # None = all; or ["cfme"], ["neurostring"]
DEVICES      = None                  # None = all ids; or ["2-2", "3-2"]
TIMEPOINTS   = None                  # None = all; or day integers e.g. [0, 4, 6]
NYQUIST_TOP_FREQS = 15               # Nyquist zoom: show only the N highest in-band frequencies
FSCV_VSHOW   = (0.0, 1.0)            # FSCV loop x-range drawn (full sweep by default)
FSCV_VSCALE  = (0.5, 0.9)            # y-autoscale sub-window (frames the peak)
FSCV_ANODIC_ONLY = True              # True = rising sweep only; False = full loop (anodic + cathodic)

DEVICES = IDX.devices(device_types=DEVICE_TYPES, devices=DEVICES)
print(f"analyzing {len(DEVICES)} device(s): " + ", ".join(IDX.title(d) for d in DEVICES)
      + (f"  |  timepoints {TIMEPOINTS}" if TIMEPOINTS is not None else "  |  all timepoints"))

## 1. EIS spectra — Bode −phase, |Z|, Nyquist (all timepoints)
Replicate-averaged EIS per channel, every measured timepoint overlaid, drawn
**Bode −phase → Bode |Z| → Nyquist**.

> **How EIS replicates are averaged:** in the **rectangular** form — the mean of `Z'` and the
> mean of `Z''` are taken **at each frequency** (`np.mean(z_real)`, `np.mean(z_imag)`), then
> `|Z|` and phase are derived from those averaged components. It is *not* an average of `|Z|`
> or phase directly. (Section 3 shows the per-replicate spread.)

- **Bode −phase:** `−phase(Z)` vs frequency; **red dashed** at `−phase = 0`; **navy dashed**
  (one per timepoint, same fade) at each timepoint's **inductive-onset frequency** (lowest `f`
  with `Im(Z) ≥ 0`). Above it the interface reads inductive and **fails EIS.1**.
- **Bode |Z|:** `|Z|` vs frequency, with the same per-timepoint navy onset lines.
- **Nyquist:** the `NYQUIST_TOP_FREQS` **highest** frequencies; points at/above the onset
  (EIS.1-failing) are **red**; **black dashed** lines mark `Z' = 0` and `−Z'' = 0`.

The onset table + `BAND` recommendation follow the plots.

In [ ]:
for dev in DEVICES:
    plot_eis(IDX, dev, "phase", timepoints=TIMEPOINTS)
    plot_eis(IDX, dev, "mag", timepoints=TIMEPOINTS)
    plot_eis(IDX, dev, "nyquist", nyq_top=NYQUIST_TOP_FREQS, timepoints=TIMEPOINTS)

### Inductive onset & recommended `BAND`
For each selected channel-timepoint, the inductive-onset frequency (lowest `f` with `Im(Z) ≥ 0`).
**Two distinct quantities — the cell keeps them separate:**

- **`BAND` (in use)** — what *this* run actually uses (set in the parameters cell, and applied to
  the section-2 plots). It is either a **manual** window you set in the shared config, or `"auto"`
  (which resolves to `recommended_band`). It is **not** overwritten here.
- **`RECOMMENDED`** — `recommended_band(ROOT)`, computed data-driven across **all data at `ROOT`**
  (global — *not* just the selected devices). Its upper bound is the highest measured frequency
  strictly below the **10th-percentile onset**, so the lowest-onset ~10% of channels fall in-band
  and **fail EIS.1** (`Im(Z) < 0` at *every* in-band frequency) as genuinely poor electrodes rather
  than a few outliers dragging the band down (`onset_percentile=0` = strict min, nobody inductive
  in-band). This is a **suggestion only**; it does not change your `BAND`.

The onset table + per-device-type distribution below are for the **selected devices**; `RECOMMENDED`
is **global**. Don't move the band per timepoint (onset drift over time is real degradation the
model should see). Set your chosen `BAND` in the "Save shared config" cell so the dashboard/discovery
match.

In [ ]:
from electropycal.data.inventory import recommended_band
onset_tbl = onset_table(IDX, DEVICES, timepoints=TIMEPOINTS)

RECOMMENDED = recommended_band(ROOT)          # data-driven, GLOBAL across ALL data at ROOT
_src = "auto -> recommended_band" if IDX.cfg.band == "auto" else "MANUAL config"
print(f"BAND IN USE = ({IDX.band[0]:.0f}, {IDX.band[1]:.0f}) Hz   [source: {_src}]")
print(f"RECOMMENDED = ({RECOMMENDED[0]:.0f}, {RECOMMENDED[1]:.0f}) Hz   "
      f"[recommended_band(ROOT): 10th-percentile onset, GLOBAL across all data at ROOT]")

_sel = onset_tbl["inductive_onset_frequency_Hz"].dropna() if len(onset_tbl) else pd.Series(dtype=float)
_fail = int((_sel <= IDX.band[1]).sum())
print(f"\nSelected: {len(_sel)} of {len(onset_tbl)} raw channel-timepoints go inductive; "
      f"{_fail} have onset <= BAND upper ({IDX.band[1]:.0f} Hz) -> would fail EIS.1 under BAND.")
with pd.option_context("display.max_rows", None):
    display(onset_tbl)

### Inductive onset frequency vs timepoint
Per device, one panel per channel: the **inductive-onset frequency** (log Hz) across timepoints in
**true day spacing**. The **red dashed** line is the current `BAND` upper bound and the shaded band
above it is the inductive / **EIS.1-fail** zone — a point in the red zone means that channel-timepoint
goes inductive in-band. A **downward drift** over time is the interface degrading. Channels with no
inductive onset in range (fully capacitive) have no marker at that timepoint.

In [ ]:
for dev in DEVICES:
    plot_inductive_onset_vs_time(IDX, dev, band=IDX.band, timepoints=TIMEPOINTS)

### Save the shared config (set once here)
Persist the `BAND` **and** the QC / extraction parameters to
`<ROOT>/electropycal_analysis_config.json` so that **`quality_filtering_dashboard`** and
**`discovery_checkpointed`** read the *same* values — no manual re-typing, no drift. Set
`CFG.band` to the recommendation above (or your own window, or `"auto"` to keep it data-driven),
tune any other field, then run this cell **once**. Skip it to leave the existing shared config
untouched. The library functions still take explicit parameters — this only feeds the notebooks.

In [ ]:
from electropycal.analysis_config import save_analysis_config
# Band the OTHER notebooks will use. Finalized value (study decision 4) = (2, 2000) Hz.
# recommended_band(ROOT) is the data-driven onset window; "auto" re-derives it per notebook.
IDX.cfg.band = (2.0, 2000.0)              # finalized; or recommended_band(ROOT), or "auto"
# tune any shared QC/extraction PARAMETER before saving (data selection is NOT saved):
# IDX.cfg.peak_method = "direct"; IDX.cfg.min_norm_snr = 3.0; IDX.cfg.monotonic_r_min = 0.6
_p = save_analysis_config(ROOT, IDX.cfg)
print(f"saved shared config -> {_p}")
print(f"  band={IDX.cfg.band}  peak={IDX.cfg.peak_method}  mono_tol={IDX.cfg.mono_tol}  "
      f"min_snr={IDX.cfg.min_norm_snr}  r_min={IDX.cfg.monotonic_r_min}  gate_on={IDX.cfg.gate_on}")
print("  re-run the cells above with RawSpectraIndex(ROOT) to view EIS in the saved band.")

## 2. EIS after applying `BAND`
The **same `BAND`** is applied to every channel-timepoint (a static window, not per-spectrum),
so the retained frequency grid is identical across sensors — required for consistent EIS
features in discovery. Drawn in the same order (**−phase → |Z| → Nyquist**); Nyquist shows the
`NYQUIST_TOP_FREQS` highest **in-band** frequencies with the `Z'=0` / `−Z''=0` guides.

In [ ]:
for dev in DEVICES:
    plot_eis(IDX, dev, "phase", band=IDX.band, timepoints=TIMEPOINTS)
    plot_eis(IDX, dev, "mag", band=IDX.band, timepoints=TIMEPOINTS)
    plot_eis(IDX, dev, "nyquist", band=IDX.band, nyq_top=NYQUIST_TOP_FREQS, timepoints=TIMEPOINTS)

## 3. EIS replicate spread — Z′ and Z″ vs frequency
Per channel, the **individual replicate** spectra (thin) with the replicate **mean ± s.d.**
band, one group per timepoint (older faded → latest solid). Two figures: **Z′ (real)** vs
frequency, then **Z″ (imaginary)** vs frequency — the two components that are averaged
(rectangularly, per section 1) to form the spectra used everywhere else. This is the one EIS
view that is *not* pre-averaged; use it to judge measurement reliability. The band is the s.d.
across replicates at each frequency. The **stats table** below lists `Z'` and `Z''` (= Im(Z))
mean ± s.d. per `(device, date, timepoint, channel, frequency)`.

In [ ]:
for dev in DEVICES:
    plot_eis_replicate_spread(IDX, dev, "zr", timepoints=TIMEPOINTS)
    plot_eis_replicate_spread(IDX, dev, "zi", timepoints=TIMEPOINTS)

In [ ]:
eis_spread_wide = eis_spread_table(IDX, DEVICES, timepoints=TIMEPOINTS, wide=True)
print(f"{len(eis_spread_wide)} rows (device x channel x timepoint); frequency (Hz) across columns "
      "- scroll right. Long form: eis_spread_table(IDX, DEVICES).")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", None):
    display(IDX.label_devices(eis_spread_wide.reset_index()) if len(eis_spread_wide) else eis_spread_wide)

## 4. FSCV raw I–V loops (per concentration)
One figure **per concentration** (`FSCV_CONCS`, default all — including the 0 nM background),
channels as panels, timepoints overlaid. Each loop is the replicate-average of the first
`REPLICATES` cycles — **averaged by taking the mean current at each sweep sample** (same `V`
position), i.e. `sum(I)/REPLICATES` with the voltage axis from the first cycle. The DA
oxidation window (0.6–0.8 V) is shaded.

**Shown raw / unsmoothed.** `electropycal` Savitzky-Golay–smooths the raw voltammogram to remove
hardware noise **by default** — but it does so from **background subtraction
onward** (`FSCV_SMOOTH_WINDOW`, applied to signal and background before subtracting). This cell
shows the signal *as measured* so you can see the noise that smoothing removes; sections 5–7
apply it and therefore match what discovery computes.

In [ ]:
for dev in DEVICES:
    plot_fscv_loops(IDX, dev, "raw", show_window=None, scale_window=None, anodic_only=False, timepoints=TIMEPOINTS)

## 5. FSCV background-subtracted `i − i_bg` (per concentration)
Same per-concentration layout, now the **background-subtracted** cycle `i − i_bg` (background =
that timepoint's 0 nM file). By default only the **anodic (rising) sweep** is drawn
(`FSCV_ANODIC_ONLY`) — the faradaic half feature extraction uses — which makes the oxidation
peak clearer. The x-axis shows `FSCV_VSHOW` (default 0.5–1.0 V) but the **y-axis autoscales to
`FSCV_VSCALE`** (default 0.5–0.9 V) per panel — so you see context past the peak while the
scale still frames the DA peak (otherwise the large swing near the switching potential flattens
it). Set the two windows equal to scale exactly to what's shown. The dashed **black** line marks
`i − i_bg = 0` (always in view); a **dot on each trace** (same color/fade) marks that timepoint's
**peak** `(V_peak, I_peak)`, located by electropycal in `PEAK_WINDOW` (default **0.4–1.0 V** —
widened from the shaded 0.6–0.8 V DA region because real DA peaks are broad and crest above
0.8 V; the upper bound stays below the ~1.0 V+ switching surge).

> **Large values approaching ~1.0 V and up** are the **anodic-limit / switching-potential**
> region (solvent/electrode background oxidation + switching capacitance): the dosed and 0 nM
> cycles both surge there and don't cancel under subtraction, and NormIpeak's division by `i_bg`
> amplifies it. It is **non-faradaic and outside the 0.6–0.8 V DA window** (and the noise-floor
> window), so feature extraction ignores it — it only appears in the plot when the shown window
> reaches the anodic limit. Keep `FSCV_VSHOW` ≤ ~1.0 V to stay clear of it.

0 nM is the background
reference (`i − i_bg ≡ 0`) so it is skipped.

**Smoothed (electropycal default).** Signal and background are Savitzky-Golay smoothed
(`FSCV_SMOOTH_WINDOW`, `FSCV_SMOOTH_POLY`) **before** subtraction — exactly as
`extract_dataset` does — so this is the de-noised trace discovery sees. (Set
`FSCV_SMOOTH_WINDOW = 0` to compare against the raw section 4. The only other transform is a
linear baseline `detrend`, off by default — a slope removal, not a smoothing filter.)

In [ ]:
for dev in DEVICES:
    plot_fscv_loops(IDX, dev, "bgsub", show_window=FSCV_VSHOW, scale_window=FSCV_VSCALE,
                    anodic_only=FSCV_ANODIC_ONLY, mark_clipped=False, timepoints=TIMEPOINTS)
    # set mark_clipped=True to flag edge-clipped peaks (peak_area under-estimate); off by default (busy).

## 6. FSCV NormIpeak-normalized loop (per concentration)
Same layout, now the **background-normalized** cycle `(i − i_bg) / i_bg` — the *NormIpeak
loop*. The scalar **NormIpeak** used downstream is this curve's peak height at the oxidation
potential `V_ox` (in the 0.6–0.8 V DA window), divided by the background there.

`PEAK_METHOD` selects how that scalar peak height is defined (the **loop drawn is the same**):
- **`"direct"`** (default) — background-subtracted current at its in-window maximum (`V_ox`);
  for this method NormIpeak equals the plotted curve's value at `V_ox`. Robust to broad peaks.
- **`"chord"`** — the height above the straight chord between the window endpoints;
  isolates a sharp peak on a curved baseline, under-counts a broad hump.

**Smoothed** (electropycal default): signal and background Savitzky-Golay smoothed
(`FSCV_SMOOTH_WINDOW`, `FSCV_SMOOTH_POLY`) before subtracting/normalizing. **Anodic sweep only**
(`FSCV_ANODIC_ONLY`); x shows `FSCV_VSHOW`, y autoscales per panel to `FSCV_VSCALE`. Dashed
**black** = NormIpeak 0; a **dot on each trace** marks that timepoint's peak `(V_peak, NormIpeak)`
(found in `PEAK_WINDOW`). Traces are replicate-averaged.

In [ ]:
for dev in DEVICES:
    plot_fscv_loops(IDX, dev, "norm", show_window=FSCV_VSHOW, scale_window=FSCV_VSCALE,
                    anodic_only=FSCV_ANODIC_ONLY, timepoints=TIMEPOINTS)

## 7. NormIpeak vs concentration (dose-response)
Per channel (panel), the scalar **NormIpeak** (`PEAK_METHOD`) vs concentration, one series per
timepoint (older faded → latest solid). **Individual replicate** NormIpeak values are shown as
points; the connecting line is the replicate **mean ± s.d.** — so this is the one place the
replicates are *not* pre-averaged. Each replicate NormIpeak is computed with the same
Savitzky-Golay smoothing (`FSCV_SMOOTH_WINDOW`) electropycal applies. A rising, monotone trend is what
FSCV.1 checks. Each panel autoscales independently (dynamic y).

The **stats table** below reports, per `(device, date, timepoint, channel, concentration)`:
`Vpeak_mean` / `Vpeak_std` (oxidation-peak voltage), raw `I_peak` (signal at V_peak), the
background-subtracted `I_peak` (`i − i_bg` at V_peak, = the NormIpeak numerator), and `NormIpeak`
— each reported as replicate **mean ± s.d.** (the mean/std are taken **across the replicate cycles
of that one concentration**). All are computed on the **Savitzky-Golay-smoothed** signal (electropycal
default), using `PEAK_METHOD` to locate the peak.

The **second plot** is the **reproducibility SNR — `NormIpeak / std(NormIpeak across replicate
cycles)`** — vs concentration. The **line** is `mean/σ` (the reproducibility SNR electropycal's
`repeatability_snr` / `acceptance="snr"` use), and the **points** are the individual replicates
normalized by `σ` so you can see the cycle-to-cycle spread behind each value; the error bars are the
normalized replicate s.d. The **red dashed** line is `MIN_NORM_SNR` (a dose passes iff its line sits
**above** it). This asks whether the peak is *reproducible across cycles*, which is what makes a
dose-response "clearly visible" — on real data good channels sit ~6× here, whereas the old RMS
noise-floor SNR sat near 1× even for obvious peaks (the RMS floor is dominated by the
background-subtraction residual, not random noise), which is why that plot was retired.

A third plot shows **`V_peak` vs concentration** in the same format (replicate points +
mean ± s.d., timepoint fade). The gray band is the **ideal** DA oxidation window (0.6–0.8 V) and
the **crimson** bands at the top/bottom are the *edge-pinned* zones (within `PEAK_EDGE_TOL` of a
`PEAK_WINDOW` bound). `V_peak` sitting in the gray band (and stable across dose/time) is what you
want; points landing in a crimson band mean the peak pinned to the search boundary — the true peak
likely lies outside `PEAK_WINDOW`, so its height/NormIpeak is a window artifact. This is the failure
the **F.pk** check (`quality_filtering_dashboard`) flags and that low DA concentrations, with their
broad, negatively-shifted peaks, are most prone to.

In [ ]:
for dev in DEVICES:
    plot_dose_response(IDX, dev, timepoints=TIMEPOINTS)

In [ ]:
for dev in DEVICES:
    plot_snr_vs_conc(IDX, dev, timepoints=TIMEPOINTS)

In [ ]:
for dev in DEVICES:
    plot_vpeak_vs_conc(IDX, dev, timepoints=TIMEPOINTS)

In [ ]:
dose_stats_tbl = dose_stats_table(IDX, DEVICES, timepoints=TIMEPOINTS)
print(f"replicate mean/std of V_peak & raw/bgsub/Norm Ipeak (smoothed, method={IDX.peak_method!r}).")
with pd.option_context("display.max_rows", None):
    display(IDX.label_devices(dose_stats_tbl))

### Peak-window edge-pinning rate by concentration
For each concentration: `n` measured `(device, channel, timepoint)`, then for each edge the count,
percent, and the **list of the pinned channel-timepoints** — split into the **lower** edge (peak
at/below `PEAK_WINDOW[0]` — the low-concentration failure), the **upper** edge (at/above
`PEAK_WINDOW[1]`), and **any** (either). A high lower-edge rate at low concentrations is the signal
that `PEAK_WINDOW`'s lower bound is clipping real, negatively-shifted DA peaks — and the lists name
exactly which sensors so you can inspect them. Use this to choose the window from your own data.

In [ ]:
edge_tbl = edge_pinning_table(IDX, DEVICES, timepoints=TIMEPOINTS)
print("V_peak pinned at a PEAK_WINDOW bound: 'lower' = low-conc clipping at the 0.4 V bound, "
      "'upper' = pinned at 1.0 V; chan-tp = 'dev chN Dt'.")
with pd.option_context("display.max_colwidth", None):
    display(edge_tbl)

## 8. NormIpeak & V_peak vs time (temporal drift, per concentration)
The two scalar readouts tracked **over time** — the *temporal* view of sections 6–7 (which put
dose on the x-axis). Here **time is the x-axis, in true day spacing** (`D0`, `D1`, … at their real
day positions), channels are panels, and **each concentration is its own trace**, colored by a
**plasma** gradient (dark = low dose → bright = high dose). Individual replicate values are shown as
points; the connecting **mean ± s.d.** line is taken over the replicate cycles at each timepoint
(same Savitzky-Golay smoothing / `PEAK_METHOD` as sections 6–7). This is the drift that recalibration
has to model.

- **NormIpeak vs time** — the response feature discovery predicts. Higher-dose traces should sit
  above lower-dose ones (dose ordering preserved), and each trace's motion across timepoints is the
  per-sensor temporal drift. Dashed **black** = NormIpeak 0.
- **V_peak vs time** — the oxidation-peak voltage over time. Gray band = the ideal DA window
  (0.6–0.8 V); **crimson** bands = the edge-pinned zones (within `PEAK_EDGE_TOL` of a `PEAK_WINDOW`
  bound). A stable V_peak inside the gray band across time is healthy; drift toward an edge flags a
  degrading / clipped peak.

In [ ]:
for dev in DEVICES:
    plot_vs_time(IDX, dev, "norm", timepoints=TIMEPOINTS)

In [ ]:
for dev in DEVICES:
    plot_vs_time(IDX, dev, "vpeak", timepoints=TIMEPOINTS)

### 8.1 Repeatability SNR vs time — is the sensor still responsive?
`NormIpeak / std(replicates)` per channel-timepoint, one plasma trace per concentration, **crimson
dashed = `MIN_NORM_SNR` (the reproducibility floor, default 3)**. This is the same quantity the
per-dose QC reliability check gates on. Use it to judge **when to stop measuring a device**: a channel
whose SNR has decayed toward the floor across *all* concentrations is measurable but no longer
reproducibly responsive — flat NormIpeak at the noise level — even if EIS still passes. Higher-dose
traces sitting well above the floor = a healthy, still-responsive sensor.

In [ ]:
for dev in DEVICES:
    plot_vs_time(IDX, dev, "snr", timepoints=TIMEPOINTS)